# How much impurity a feature removed

Each split is credited to the feature it used. The credit is the size of the node times the gain, divided by the size of the whole training set:

$$
\frac{n_{\text{node}}}{n} \cdot \operatorname{Gain}
$$

On the eight parts, length cuts first. The node is the whole set, $n_{\text{node}}/n = 1$, and the gain is $1/6$. Length's credit is $1/6$.

Weight then cuts the left child. That child holds $6$ of the $8$ parts, and the gain inside it is $4/9$.

$$
\frac{6}{8} \cdot \frac{4}{9} = \frac{1}{3}
$$

Weight's credit is $1/3$. The two credits add to the parent's original Gini:

$$
\frac{1}{6} + \frac{1}{3} = \frac{1}{2}
$$

Divide by that total to put the credits on a $0$-to-$1$ scale.

$$
\text{length} = \frac{1/6}{1/2} = \frac{1}{3},
\qquad
\text{weight} = \frac{1/3}{1/2} = \frac{2}{3}
$$

Weight receives more credit because its cut removed the larger share of the impurity, even though length asked the first question.


In [1]:
# Credits add back to the parent Gini. Weight receives 2/3 of the credit.
from fractions import Fraction

length_credit = Fraction(1, 6)
weight_credit = Fraction(6, 8) * Fraction(4, 9)
assert weight_credit == Fraction(1, 3)
assert length_credit + weight_credit == Fraction(1, 2)
assert length_credit / Fraction(1, 2) == Fraction(1, 3)
assert weight_credit / Fraction(1, 2) == Fraction(2, 3)
print("length", Fraction(1, 3), "weight", Fraction(2, 3))


length 1/3 weight 2/3


## The tie swaps the story

Weight $\le 3$ had the same root gain, $1/6$. The tie-break chose length only because length is listed first. If weight had been listed first, weight would take the credit $1/6$ and length would take the later credit $1/3$. The normalized numbers would swap: weight $1/3$, length $2/3$.

The eight parts did not change. The importance ranking did. A ranking that comes from a tied root is not a property of the table until you publish the tie-break.


In [2]:
# Swap the feature order and the normalized credits swap with it.
from fractions import Fraction

def credits(first, second):
    return {first: Fraction(1, 6), second: Fraction(1, 3)}

length_first = credits("length", "weight")
weight_first = credits("weight", "length")
assert length_first["weight"] == weight_first["length"]
print("length asked first", length_first)
print("weight asked first", weight_first)


length asked first {'length': Fraction(1, 6), 'weight': Fraction(1, 3)}
weight asked first {'weight': Fraction(1, 6), 'length': Fraction(1, 3)}


## A pitfall

A feature with credit $0$ was not asked. It might still separate the parts after some other question. XOR is the sharp case: the first cut has gain $0$, so a rule that stops there assigns both features credit $0$, even though a depth-two tree separates the four points. Credit summarizes the tree you grew, not every tree you might have grown.

## What to carry forward

Credit is $(n_{\text{node}}/n)$ times gain. With length asked first, the credits are $1/6$ and $1/3$, which normalize to $1/3$ and $2/3$. The tied root can swap those two numbers.
